# 00. Imports & Functions

## Imports

In [10]:
from convert_lif import normalize, save_out
import os
import aicsimageio
import pandas as pd
import sys
import zarr
import numpy as np
import time

import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from skfda.representation.basis import BSplineBasis
from skfda.misc.regularization import L2Regularization
from skfda.misc.operators import LinearDifferentialOperator
from skfda.preprocessing.smoothing import BasisSmoother
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold
from collections import defaultdict
import skfda

from sklearn.metrics import mean_squared_error, r2_score

import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import KFold, GroupKFold
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.distributed
import pandas as pd
import numpy as np
from ABRA_012926 import interpolate_and_smooth, CNN, plot_wave, calculate_and_plot_wave, plot_waves_single_frequency, arfread, get_str, calculate_hearing_threshold, all_thresholds, peak_finding
import warnings
from sklearn.preprocessing import StandardScaler,MinMaxScaler
warnings.filterwarnings('ignore')
import os
import io
import re
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import KFold, cross_val_score
from sklearn.model_selection import GroupShuffleSplit
import os
import struct
import datetime
# from skfda import FDataGrid
# from skfda.preprocessing.dim_reduction import FPCA
from sklearn.preprocessing import StandardScaler,MinMaxScaler
import numpy as np
import pandas as pd
import struct
import matplotlib.pyplot as plt
import torch
from scipy.ndimage import gaussian_filter1d
import torch.nn as nn
# from tensorflow.keras.models import load_model
from scipy.interpolate import CubicSpline
from scipy.signal import find_peaks
# from tensorflow.keras.preprocessing.image import ImageDataGenerator
from pathlib import Path
import torch
from PIL import Image
import random
# import tensorflow as tf
import warnings
# warnings.filterwarnings('ignore')
# import pytorch libraries
%matplotlib inline
import torch 
import torch.autograd as autograd 
import torch.nn as nn 
import torch.nn.functional as F
import torch.optim as optim
import numpy as np4
from sklearn.metrics import r2_score
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import KFold, cross_val_score

ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

## ABRA Functions

In [11]:
def latency_all_peaks(highest_peaks, y_values, time_scale):
    latencies = []
    num_peaks = highest_peaks.size
    if num_peaks > 0:  # Check if highest_peaks is not empty
        for n in range(num_peaks): # SHOULD be 5 but there are cases where there are less. Will handle in later loops
            lat = highest_peaks[n] * (time_scale / len(y_values)) # Based on ABRA logic
            latencies.append(lat)
        return latencies
    else:
        print("No peaks detected. Check input data")
        return None

In [12]:
def full_interpolation(df, freq, db, time_scale=10, multiply_y_factor=1.0, units='Microvolts'):
    
    khz = df[(df['Freq(kHz)'] == freq) & (df['Level(dB)'] == db)]
    # print(khz)
    if not khz.empty:
        index = khz.index.values[0]
        final = df.loc[index, '0':].dropna()
        final = pd.to_numeric(final, errors='coerce').dropna()

        target = int(244 * (time_scale / 10))
        
        # Process the wave as in calculate_and_plot_wave
        y_values = interpolate_and_smooth(final, target)

        # print(f"Interpolated y_values: {y_values[:5]}")
        # print(f"Any NaNs? {np.isnan(y_values).any()}")

        if final.empty:
            print(f"Warning: Empty waveform for {freq}kHz @ {db}dB")
            return np.full((1, 244), np.nan)
        
        # Apply scaling factor
        y_values *= multiply_y_factor
        
        # Handle units conversion if needed
        if units == 'Nanovolts':
            y_values /= 1000
            
        # Generate normalized version for peak finding
        y_values_fpf = interpolate_and_smooth(y_values[:244])
        
        # Standardize and normalize for peak finding, exactly as in the original
        flattened_data = y_values_fpf.flatten().reshape(-1, 1)
        scaler = StandardScaler()
        standardized_data = scaler.fit_transform(flattened_data)
        min_max_scaler = MinMaxScaler(feature_range=(0, 1))
        scaled_data = min_max_scaler.fit_transform(standardized_data).reshape(y_values_fpf.shape)
    
        return scaled_data

In [13]:
def peak_finding(wave):
    # Prepare waveform
    waveform=interpolate_and_smooth(wave) # Added indexing per calculate and plot wave function
    # waveform_torch = torch.tensor(waveform, dtype=torch.float32).unsqueeze(0) archived ABRA
    waveform_torch = torch.tensor(waveform, dtype=torch.float32).unsqueeze(0).unsqueeze(0) #newer ABRA
    # print(waveform_torch)
    # Get prediction from model
    outputs = peak_finding_model(waveform_torch)
    prediction = int(round(outputs.detach().numpy()[0][0], 0))
    # prediction_test = int(round(outputs.detach().numpy()[0], 0))
    # print("Model output:", outputs, "Prediction true start:", prediction)

    # Apply Gaussian smoothing
    smoothed_waveform = gaussian_filter1d(waveform, sigma=1)

    # Find peaks and troughs
    n = 18
    t = 14
    # start_point = prediction - 9 archived ABRA
    start_point = prediction - 6 #newer ABRA
    smoothed_peaks, _ = find_peaks(smoothed_waveform[start_point:], distance=n)
    smoothed_troughs, _ = find_peaks(-smoothed_waveform, distance=t)
    sorted_indices = np.argsort(smoothed_waveform[smoothed_peaks+start_point])
    highest_smoothed_peaks = np.sort(smoothed_peaks[sorted_indices[-5:]] + start_point)
    relevant_troughs = np.array([])
    for p in range(len(highest_smoothed_peaks)):
        c = 0
        for t in smoothed_troughs:
            if t > highest_smoothed_peaks[p]:
                if p != 4:
                    try:
                        if t < highest_smoothed_peaks[p+1]:
                            relevant_troughs = np.append(relevant_troughs, int(t))
                            break
                    except IndexError:
                        pass
                else:
                    relevant_troughs = np.append(relevant_troughs, int(t))
                    break
    relevant_troughs = relevant_troughs.astype('i')
    return highest_smoothed_peaks, relevant_troughs

def extract_metadata(metadata_lines):
    # Dictionary to store extracted metadata
    metadata = {}
    
    for line in metadata_lines:
        # Extract SW FREQ
        freq_match = re.search(r'SW FREQ:\s*(\d+\.?\d*)', line)
        if freq_match:
            metadata['SW_FREQ'] = float(freq_match.group(1))
        
        # Extract LEVELS
        levels_match = re.search(r':LEVELS:\s*([^:]+)', line)
        if levels_match:
            # Split levels and convert to list of floats
            metadata['LEVELS'] = [float(level) for level in levels_match.group(1).split(';') if level]
    
    return metadata

def read_custom_tsv(file_path):
    # Read the entire file
    with open(file_path, 'r', encoding='ISO-8859-1') as f:
        content = f.read()
    
    # Split the content into metadata and data sections
    metadata_lines = []
    data_section = None
    
    # Find the ':DATA' marker
    data_start = content.find(':DATA')
    
    if data_start != -1:
        # Extract metadata (lines before ':DATA')
        metadata_lines = content[:data_start].split('\n')
        
        # Extract data section
        data_section = content[data_start:].split(':DATA')[1].strip()
    
    # Extract specific metadata
    metadata = extract_metadata(metadata_lines)
    
    # Read the data section directly
    try:
        # Use StringIO to create a file-like object from the data section
        raw_data = pd.read_csv(
            io.StringIO(data_section), 
            sep='\s+',  # Use whitespace as separator
            header=None
        )
        raw_data = raw_data.T
        # Add metadata columns to the DataFrame
        if 'SW_FREQ' in metadata:
            raw_data['Freq(kHz)'] = metadata['SW_FREQ']
            # raw_data['Freq(Hz)'] = raw_data['Freq(Hz)'].apply(lambda x: x*1000)
        
        if 'LEVELS' in metadata:
            # Repeat levels to match the number of rows
            levels_repeated = metadata['LEVELS'] * (len(raw_data) // len(metadata['LEVELS']) + 1)
            raw_data['Level(dB)'] = levels_repeated[:len(raw_data)]
        
        filtered_data = raw_data.apply(pd.to_numeric, errors='coerce').dropna()
        filtered_data.columns = filtered_data.columns.map(str)

        columns = ['Freq(kHz)'] + ['Level(dB)'] + [col for col in filtered_data.columns if col.isnumeric() == True]
        filtered_data = filtered_data[columns]
        return filtered_data
    
    except Exception as e:
        print(f"Error reading data: {e}")
        return None, metadata

In [14]:
def peaks_troughs_amp_final(df, freq, db, time_scale=10, multiply_y_factor=1.0, units='Microvolts'):
    db_column = 'Level(dB)'
    
    khz = df[(df['Freq(kHz)'] == freq) & (df[db_column] == db)]
    if not khz.empty:
        index = khz.index.values[0]
        final = df.loc[index, '0':].dropna()
        final = pd.to_numeric(final, errors='coerce').dropna()

        target = int(244 * (time_scale / 10))
        
        # Process the wave as in calculate_and_plot_wave
        y_values = interpolate_and_smooth(final, target)
        
        # Apply scaling factor
        y_values *= multiply_y_factor
        
        # Handle units conversion if needed
        if units == 'Nanovolts':
            y_values /= 1000
            
        # Generate normalized version for peak finding
        y_values_fpf = interpolate_and_smooth(y_values[:244])
        
        # Standardize and normalize for peak finding, exactly as in the original
        flattened_data = y_values_fpf.flatten().reshape(-1, 1)
        scaler = StandardScaler()
        standardized_data = scaler.fit_transform(flattened_data)
        min_max_scaler = MinMaxScaler(feature_range=(0, 1))
        scaled_data = min_max_scaler.fit_transform(standardized_data).reshape(y_values_fpf.shape)
        y_values_fpf = interpolate_and_smooth(scaled_data[:244])
        
        # Find peaks using the normalized data
        highest_peaks, relevant_troughs = peak_finding(y_values_fpf)
        
        # Calculate amplitude on the processed but non-normalized data
        if highest_peaks.size > 0 and relevant_troughs.size > 0:
            # Following the same approach as in the display_metrics_table function
            first_peak_amplitude = y_values[highest_peaks[0]] - y_values[relevant_troughs[0]]
            return highest_peaks, relevant_troughs, first_peak_amplitude
    
    return None, None, None

In [15]:
# ============================================================================
# UNSUPERVISED IMAGE EMBEDDING MODELS FOR SYNAPSE CLUSTERING
# ============================================================================
# Implementation of 3 progressive approaches for learning embeddings from
# 4-channel confocal microscopy images (Myo7, GluR2, CtBP2, NF)
#
# Phase 1: Shallow CNN - Quick signal validation
# Phase 2: Convolutional Autoencoder - Better representations
# Phase 3: Variational Autoencoder (VAE) - Structured latent space (BEST)
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, DBSCAN
from sklearn.manifold import TSNE
import umap
from sklearn.metrics import silhouette_score


/opt/anaconda3/envs/manor_image_pred/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 0. Graveyard

In [7]:
from bioio import BioImage
import bioio_lif
import bioio_tifffile
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt



# def create_multi_channel_embedder(num_input_channels=4, embedding_dim=512, pretrained=True):
#     """
#     Create a CNN that accepts multi-channel input and outputs embeddings.
    
#     Args:
#         num_input_channels: Number of input channels (e.g., 4 for your LIF files)
#         embedding_dim: Size of output embedding vector
#         pretrained: Use ImageNet pretrained weights (will adapt first layer)
    
#     Returns:
#         PyTorch model
#     """
#     # Load pretrained ResNet18
#     resnet = models.resnet18(pretrained=pretrained)
    
#     # Modify first conv layer to accept custom number of channels
#     resnet.conv1 = nn.Conv2d(num_input_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
    
#     # Replace final FC layer with embedding layer
#     num_features = resnet.fc.in_features
#     resnet.fc = nn.Linear(num_features, embedding_dim)
    
#     resnet.eval()
#     return resnet


def process_image_to_embedding(img_path, embedder, return_numpy=True):
    """
    Complete pipeline: Load image → Max project → Normalize → Embed → Numpy
    
    Args:
        img_path: Path to image file
        embedder: Pre-trained embedding model
        return_numpy: If True, return numpy array; if False, return torch tensor
    
    Returns:
        Embedding as numpy array (shape: [512]) or torch tensor
    """
    # Load and max project
    img_3d = load_and_max_project(img_path)
    
    # Normalize
    img_norm = normalize_per_channel(img_3d)
    
    # Generate embedding
    embedding = get_embedding_multi_channel(img_norm, embedder)
    
    # Convert to numpy if requested
    if return_numpy:
        return embedding_to_numpy(embedding)
    else:
        return embedding

# # Step 2: Normalize each channel
# img_3d_norm = normalize_per_channel(img_3d)
# print(f"Normalized: {img_3d_norm.shape}")

# # Visualize all channels
# fig, axes = plt.subplots(1, 4, figsize=(16, 4))
# channel_names = ['CtBP2', 'GluR2', 'NF', 'Myo7']
# for i in range(4):
#     axes[i].imshow(img_3d_norm[i].numpy(), cmap='gray')
#     axes[i].set_title(f'Channel {i}: {channel_names[i]}')
#     axes[i].axis('off')
# plt.tight_layout()
# plt.show()

# # Step 3: Create embedder (do this once, reuse for all images)
# print("\n" + "="*60)
# print("STEP 2: Creating multi-channel embedder")
# print("="*60)
# embedder = create_multi_channel_embedder(num_input_channels=4, embedding_dim=512)
# print("✓ Embedder created (ResNet18-based, 4 channels → 512-D)")

# # Step 4: Generate embeddings as numpy arrays
# print("\n" + "="*60)
# print("STEP 3: Generating embeddings (as numpy arrays)")
# print("="*60)

# # Method 1: Step by step
# embedding_v1_tensor = get_embedding_multi_channel(img_3d_norm, embedder)
# embedding_v1_numpy = embedding_to_numpy(embedding_v1_tensor)

# print(f"✓ v1 embedding (numpy): shape={embedding_v1_numpy.shape}, dtype={embedding_v1_numpy.dtype}")
# print(f"  First 10 values: {embedding_v1_numpy[:10]}")

# # Method 2: All in one (cleaner for batch processing)
# embedding_v2_numpy = process_image_to_embedding(path2, embedder, return_numpy=True)

# print(f"✓ v2 embedding (numpy): shape={embedding_v2_numpy.shape}, dtype={embedding_v2_numpy.dtype}")
# print(f"  First 10 values: {embedding_v2_numpy[:10]}")

# # Step 5: Demonstrate usage in deep learning
# print("\n" + "="*60)
# print("STEP 4: Using embeddings in deep learning models")
# print("="*60)

# # Example: Create a batch of embeddings for deep learning
# all_embeddings = np.stack([embedding_v1_numpy, embedding_v2_numpy])
# print(f"Batch of embeddings: {all_embeddings.shape} (batch_size=2, embedding_dim=512)")

# # Convert back to torch for PyTorch models
# torch_batch = torch.from_numpy(all_embeddings).float()
# print(f"As PyTorch tensor: {torch_batch.shape}")

# # Example: Save embeddings to disk
# np.save('embedding_v1.npy', embedding_v1_numpy)
# np.save('embedding_v2.npy', embedding_v2_numpy)
# print(f"\n✓ Embeddings saved to .npy files")

# # Example: Load embeddings from disk
# loaded_embedding = np.load('embedding_v1.npy')
# print(f"✓ Loaded embedding: {loaded_embedding.shape}")

# print("\n" + "="*60)
# print("✓ ALL DONE! Embeddings ready for deep learning!")
# print("="*60)
# print("\nYou can use these embeddings as:")
# print("  1. Input features to any sklearn model (Random Forest, SVM, etc.)")
# print("  2. Input to PyTorch/TensorFlow neural networks")
# print("  3. Store in pandas DataFrame for analysis")


# print("  4. Save/load with np.save() and np.load()")


ModuleNotFoundError: No module named 'bioio_lif'

# 1. Non-Image Data Processing

## Initial Data Collection

In [8]:
filter1 = 128
filter2 = 32
dropout1 = 0.5
dropout2 = 0.3
dropout_fc = 0.1

# Model initialization
peak_finding_model = CNN(filter1, filter2, dropout1, dropout2, dropout_fc)
model_loader = torch.load('./models/waveI_cnn.pth')
peak_finding_model.load_state_dict(model_loader)
peak_finding_model.eval()

CNN(
  (conv1): Conv1d(1, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv1d(128, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (fc1): Linear(in_features=1952, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=1, bias=True)
  (dropout1): Dropout(p=0.5, inplace=False)
  (dropout2): Dropout(p=0.3, inplace=False)
  (dropout_fc): Dropout(p=0.1, inplace=False)
  (batch_norm1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (batch_norm2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
)

In [9]:
# del int
time_scale = 18
amp_per_freq = {'Subject': [], 'Latencies' : [], 'Freq(kHz) (x1)': [], 'Level(dB) (x2)': [], 'Amplitude (x3)':[]}
start_path = '/Users/leahashebir/Downloads/Manor_Practicum/liberman_data/abr_data/WPZ Electrophysiology'
for subject in os.listdir(start_path):
    # print("Subject:",subject)
    for fq in os.listdir(os.path.join(start_path,subject)):
        # print(fq)
        if fq.startswith('ABR') and fq.endswith('.tsv'):
            path = os.path.join(start_path,subject,fq)
            data_df = read_custom_tsv(path)
            # print(data_df)
            freqs = data_df['Freq(kHz)'].unique().tolist()
            levels = data_df['Level(dB)'].unique().tolist()
            for freq in freqs:
                for lvl in levels:
                    highest_peaks, y_values, amp = peaks_troughs_amp_final(df=data_df, freq=freq, db=lvl, time_scale=time_scale)
                    latencies = latency_all_peaks(highest_peaks, y_values, time_scale)
                    if len(latencies) < 5:
                        print(subject, freq , latencies)
                        continue
                    amp_per_freq['Subject'].append(subject)
                    amp_per_freq['Freq(kHz) (x1)'].append(freq)
                    amp_per_freq['Level(dB) (x2)'].append(lvl)
                    amp_per_freq['Amplitude (x3)'].append(amp)
                    amp_per_freq['Latencies'].append(latencies)
        else:
            pass


amp_df_full = pd.DataFrame(data=amp_per_freq)

raw_synapse_counts = pd.read_excel('/Users/leahashebir/Downloads/Manor_Practicum/liberman_data/WPZ Ribbon and Synapse Counts.xlsx')
# raw_synapse_counts = raw_synapse_counts.mask(lambda x: x.isnull()).dropna() # old approach 
raw_synapse_counts = raw_synapse_counts.mask(lambda x: x.isna(),0)
raw_synapse_counts['Synapses to IHC (y1)'] = raw_synapse_counts.iloc[:,6]
raw_synapse_counts['vx (x4)'] = raw_synapse_counts['vx']
raw_synapse_counts.drop(columns=['vx'], inplace=True)
raw_synapse_counts.rename(columns={'Freq':'Freq(kHz) (x1)'}, inplace=True)
raw_synapse_counts.rename(columns={'Case':'Subject'}, inplace=True)

WPZ62 45.2 [np.float64(486.0), np.float64(762.0), np.float64(1110.0), np.float64(1350.0)]


In [10]:
# Version 1 - values per vx

paired = amp_df_full.join(raw_synapse_counts.set_index(['Subject', 'Freq(kHz) (x1)']), on=['Subject', 'Freq(kHz) (x1)'])
# slice = paired[paired['Subject']=='WPZ174'][['Subject', 'Freq(kHz) (x1)', 'Level(dB) (x2)', 'Amplitude (x3)', 'vx (x4)','Synapses to IHC (y1)', 'IHCs (y2)']]
final = paired[['Subject', 'Latencies', 'Freq(kHz) (x1)', 'Level(dB) (x2)', 'Amplitude (x3)', 'vx (x4)','Synapses to IHC (y1)', 'IHCs']]
final_clean = final.dropna()

# adding in the strain feature
strains = pd.read_excel('/Users/leahashebir/Downloads/Manor_Practicum/liberman_data/WPZ Mouse groups.xlsx')
final_clean_strained = final_clean.join(strains.set_index('ID#'), on='Subject')
final_clean_strained['Strain'] = final_clean_strained['Strain'].str.strip()
final_clean_strained = final_clean_strained.rename(columns={'Strain': 'Strain (x5)'})
final_clean_strained = final_clean_strained.dropna()
final_clean_strained = final_clean_strained[['Subject', 'Latencies', 'Freq(kHz) (x1)', 'Level(dB) (x2)', 'Amplitude (x3)', 'vx (x4)', 'Strain (x5)', 'Synapses to IHC (y1)', 'Group']]

final_clean_strained_grouped = final_clean_strained.copy()
final_clean_strained_grouped['Group - dB'] = final_clean_strained_grouped['Group'].apply(lambda x: x.split(' ')[0] if x.split(' ')[0].endswith('dB') else 'Control')
final_clean_strained_grouped['Group - Time Elapsed'] = final_clean_strained_grouped['Group'].apply(lambda x: x.split(' ')[1] if x.split(' ')[1].endswith(('h', 'wks', 'w')) else x.split(' ')[0])
final_clean_strained_grouped.head()

final_clean_strained_grouped_pos = final_clean_strained_grouped.copy()
final_clean_strained_grouped_pos['Amplitude (x3)'] = final_clean_strained_grouped['Amplitude (x3)'].apply(lambda x: 0 if x < 0 else x)

final_clean_strained_grouped_pos_cleangroup = final_clean_strained_grouped_pos.copy()
final_clean_strained_grouped_pos_cleangroup['Group'] = final_clean_strained_grouped_pos_cleangroup['Group'].apply(lambda x: x.strip())

final_clean_strained_grouped_pos_cleangroup.head()
final_clean_strained_grouped_pos_cleangroup_vs = final_clean_strained_grouped_pos_cleangroup.copy()
final_clean_strained_grouped_pos_cleangroup_vs['Group - dB (x6)'] = final_clean_strained_grouped_pos_cleangroup_vs['Group - dB']
# final_clean_strained_grouped_pos_cleangroup_vs['Group - Time Elapsed (x7)'] = final_clean_strained_grouped_pos_cleangroup_vs['Group - Time Elapsed']
final_clean_strained_grouped_pos_cleangroup_vs = final_clean_strained_grouped_pos_cleangroup_vs[['Subject','Latencies', 'Freq(kHz) (x1)', 'Level(dB) (x2)', 'Amplitude (x3)',
       'vx (x4)', 'Strain (x5)','Group - dB (x6)', 'Group - Time Elapsed', 'Group','Synapses to IHC (y1)']]

def split_on_number(input_string):
    return re.findall(r"[A-Za-z]+|\d+", input_string)

hrs_week = 24*7

final_clean_strained_grouped_pos_cleangroup_vs_timed = final_clean_strained_grouped_pos_cleangroup_vs.copy()
final_clean_strained_grouped_pos_cleangroup_vs_timed['Group - dB (x6)'] = final_clean_strained_grouped_pos_cleangroup_vs_timed['Group - dB (x6)'].apply(lambda x: '0dB' if x == 'Control' else x)
final_clean_strained_grouped_pos_cleangroup_vs_timed['Group - dB (x6)'] = final_clean_strained_grouped_pos_cleangroup_vs_timed['Group - dB (x6)'].apply(split_on_number)
final_clean_strained_grouped_pos_cleangroup_vs_timed['Group - dB (x6)'] = final_clean_strained_grouped_pos_cleangroup_vs_timed['Group - dB (x6)'].apply(lambda x: x[0])
final_clean_strained_grouped_pos_cleangroup_vs_timed['Group - dB (x6)'] = final_clean_strained_grouped_pos_cleangroup_vs_timed['Group - dB (x6)'].apply(lambda x: int(x.strip()))
final_clean_strained_grouped_pos_cleangroup_vs_timed['Group - Time Elapsed - Split'] = final_clean_strained_grouped_pos_cleangroup_vs_timed['Group - Time Elapsed'].apply(split_on_number)
final_clean_strained_grouped_pos_cleangroup_vs_timed['Group - Time Elapsed - Magn.'] = final_clean_strained_grouped_pos_cleangroup_vs_timed['Group - Time Elapsed - Split'].apply(lambda x: x[0])
final_clean_strained_grouped_pos_cleangroup_vs_timed['Group - Time Elapsed - Magn.'] = final_clean_strained_grouped_pos_cleangroup_vs_timed['Group - Time Elapsed - Magn.'].apply(lambda x: int(x.strip()))
final_clean_strained_grouped_pos_cleangroup_vs_timed['Group - Time Elapsed - Unit'] = final_clean_strained_grouped_pos_cleangroup_vs_timed['Group - Time Elapsed - Split'].apply(lambda x: x[1])
final_clean_strained_grouped_pos_cleangroup_vs_timed['Group - Time Elapsed - Unit'] = final_clean_strained_grouped_pos_cleangroup_vs_timed['Group - Time Elapsed - Unit'].apply(lambda x: "wks" if x == 'w' else x)
final_clean_strained_grouped_pos_cleangroup_vs_timed['Group - Hours Elapsed (x7)'] = final_clean_strained_grouped_pos_cleangroup_vs_timed.apply(lambda row: row['Group - Time Elapsed - Magn.']* hrs_week if row['Group - Time Elapsed - Unit'] == 'wks' else row['Group - Time Elapsed - Magn.'], axis = 1)

final_clean_strained_grouped_pos_cleangroup_vs_timed_avgvx_encoded = final_clean_strained_grouped_pos_cleangroup_vs_timed.copy()
final_clean_strained_grouped_pos_cleangroup_vs_timed_avgvx_encoded['Strain encoded (x5)'] = final_clean_strained_grouped_pos_cleangroup_vs_timed_avgvx_encoded['Strain (x5)'].apply(lambda x: 0 if x=='C57B6' else 1)
final_clean_strained_grouped_pos_cleangroup_vs_timed_avgvx_encoded.head()


,Subject,Latencies,Freq(kHz) (x1),Level(dB) (x2),Amplitude (x3),vx (x4),Strain (x5),Group - dB (x6),Group - Time Elapsed,Group,Synapses to IHC (y1),Group - Time Elapsed - Split,Group - Time Elapsed - Magn.,Group - Time Elapsed - Unit,Group - Hours Elapsed (x7),Strain encoded (x5)
0,WPZ145,"[337.5, 589.5, 724.5, 895.5, 1071.0]",45.2,70.0,0.033579,v1,C57B6,98,8wks,98dB 8wks post,8.750000,"[8, wks]",8,wks,1344,0
0,WPZ145,"[337.5, 589.5, 724.5, 895.5, 1071.0]",45.2,70.0,0.033579,v2,C57B6,98,8wks,98dB 8wks post,10.888889,"[8, wks]",8,wks,1344,0
1,WPZ145,"[266.40000000000003, 352.8, 525.6, 651.6, 824.4]",45.2,75.0,0.034262,v1,C57B6,98,8wks,98dB 8wks post,8.750000,"[8, wks]",8,wks,1344,0
1,WPZ145,"[266.40000000000003, 352.8, 525.6, 651.6, 824.4]",45.2,75.0,0.034262,v2,C57B6,98,8wks,98dB 8wks post,10.888889,"[8, wks]",8,wks,1344,0
2,WPZ145,"[252.0, 400.5, 715.5, 904.5, 1075.5]",45.2,80.0,0.154224,v1,C57B6,98,8wks,98dB 8wks post,8.750000,"[8, wks]",8,wks,1344,0


In [11]:
# Version 2 - Averaged per Vx

paired2 = amp_df_full.join(raw_synapse_counts.set_index(['Subject', 'Freq(kHz) (x1)']), on=['Subject', 'Freq(kHz) (x1)'])
# lilslice = paired[paired['Subject']=='WPZ174'][['Subject', 'Freq(kHz) (x1)', 'Level(dB) (x2)', 'Amplitude (x3)', 'vx (x4)','Synapses to IHC (y1)', 'IHCs']]
final2 = paired2[['Subject', 'Latencies', 'Freq(kHz) (x1)', 'Level(dB) (x2)', 'Amplitude (x3)', 'vx (x4)','Synapses to IHC (y1)', 'Synapses', 'IHCs']]
final_clean2 = final2.dropna()

# adding in the strain feature
strains = pd.read_excel('/Users/leahashebir/Downloads/Manor_Practicum/liberman_data/WPZ Mouse groups.xlsx')
final_clean_strained2 = final_clean2.join(strains.set_index('ID#'), on='Subject')
final_clean_strained2['Strain'] = final_clean_strained2['Strain'].str.strip()
final_clean_strained2 = final_clean_strained2.rename(columns={'Strain': 'Strain (x5)'})
final_clean_strained2 = final_clean_strained2.dropna()
final_clean_strained2 = final_clean_strained2[['Subject', 'Latencies', 'Freq(kHz) (x1)', 'Level(dB) (x2)', 'Amplitude (x3)', 'vx (x4)', 'Strain (x5)', 'Synapses to IHC (y1)', 'Group', 'Synapses', 'IHCs']]
# np.unique(final_clean_strained2['Group'])

# final_clean_70 = final_clean[final_clean['Level(dB) (x2)'] >= 70.0]
# final_clean_strained_70 = final_clean_strained[final_clean_strained['Level(dB) (x2)'] >= 70.0]
# # np.unique(final_clean['Level(dB) (x2)']) max level is 80 db
# len(final_clean), len(final_clean_70) # 10000 less data points!!!

final_clean_strained_grouped2 = final_clean_strained2.copy()
final_clean_strained_grouped2['Group - dB'] = final_clean_strained_grouped2['Group'].apply(lambda x: x.split(' ')[0] if x.split(' ')[0].endswith('dB') else 'Control')
final_clean_strained_grouped2['Group - Time Elapsed'] = final_clean_strained_grouped2['Group'].apply(lambda x: x.split(' ')[1] if x.split(' ')[1].endswith(('h', 'wks', 'w')) else x.split(' ')[0])
final_clean_strained_grouped2.head()

final_clean_strained_grouped_pos2 = final_clean_strained_grouped2.copy()
final_clean_strained_grouped_pos2['Amplitude (x3)'] = final_clean_strained_grouped2['Amplitude (x3)'].apply(lambda x: 0 if x < 0 else x)

len(final_clean_strained_grouped_pos2[final_clean_strained_grouped_pos2['Amplitude (x3)'] < 0])

final_clean_strained_grouped_pos2['Amplitude (x3)'] = final_clean_strained_grouped2['Amplitude (x3)'].apply(lambda x: 0 if x < 0 else x)

# final_clean_strained_grouped_pos[(final_clean_strained_grouped_pos['Subject'] == 'WPZ66') & (final_clean_strained_grouped_pos['Amplitude (x3)'] ==0.055901451434921576)
final_clean_strained_grouped_pos_cleangroup2 = final_clean_strained_grouped_pos2.copy()
final_clean_strained_grouped_pos_cleangroup2['Group'] = final_clean_strained_grouped_pos_cleangroup2['Group'].apply(lambda x: x.strip())
np.unique(final_clean_strained_grouped_pos_cleangroup2['Group'])

final_clean_strained_grouped_pos_cleangroup2.head()
final_clean_strained_grouped_pos_cleangroup_vs2 = final_clean_strained_grouped_pos_cleangroup2.copy()
final_clean_strained_grouped_pos_cleangroup_vs2['Group - dB (x6)'] = final_clean_strained_grouped_pos_cleangroup_vs2['Group - dB']
# final_clean_strained_grouped_pos_cleangroup_vs['Group - Time Elapsed (x7)'] = final_clean_strained_grouped_pos_cleangroup_vs['Group - Time Elapsed']
final_clean_strained_grouped_pos_cleangroup_vs2 = final_clean_strained_grouped_pos_cleangroup_vs2[['Subject',  'Latencies', 'Freq(kHz) (x1)', 'Level(dB) (x2)', 'Amplitude (x3)',
       'vx (x4)', 'Strain (x5)','Group - dB (x6)', 'Group - Time Elapsed', 'Group','Synapses to IHC (y1)', 'Synapses', 'IHCs']]

def split_on_number(input_string):
    return re.findall(r"[A-Za-z]+|\d+", input_string)

hrs_week = 24*7

final_clean_strained_grouped_pos_cleangroup_vs_timed2 = final_clean_strained_grouped_pos_cleangroup_vs2.copy()
final_clean_strained_grouped_pos_cleangroup_vs_timed2['Group - dB (x6)'] = final_clean_strained_grouped_pos_cleangroup_vs_timed2['Group - dB (x6)'].apply(lambda x: '0dB' if x == 'Control' else x)
final_clean_strained_grouped_pos_cleangroup_vs_timed2['Group - dB (x6)'] = final_clean_strained_grouped_pos_cleangroup_vs_timed2['Group - dB (x6)'].apply(split_on_number)
final_clean_strained_grouped_pos_cleangroup_vs_timed2['Group - dB (x6)'] = final_clean_strained_grouped_pos_cleangroup_vs_timed2['Group - dB (x6)'].apply(lambda x: x[0])
final_clean_strained_grouped_pos_cleangroup_vs_timed2['Group - dB (x6)'] = final_clean_strained_grouped_pos_cleangroup_vs_timed2['Group - dB (x6)'].apply(lambda x: int(x.strip()))
final_clean_strained_grouped_pos_cleangroup_vs_timed2['Group - Time Elapsed - Split'] = final_clean_strained_grouped_pos_cleangroup_vs_timed2['Group - Time Elapsed'].apply(split_on_number)
final_clean_strained_grouped_pos_cleangroup_vs_timed2['Group - Time Elapsed - Magn.'] = final_clean_strained_grouped_pos_cleangroup_vs_timed2['Group - Time Elapsed - Split'].apply(lambda x: x[0])
final_clean_strained_grouped_pos_cleangroup_vs_timed2['Group - Time Elapsed - Magn.'] = final_clean_strained_grouped_pos_cleangroup_vs_timed2['Group - Time Elapsed - Magn.'].apply(lambda x: int(x.strip()))
final_clean_strained_grouped_pos_cleangroup_vs_timed2['Group - Time Elapsed - Unit'] = final_clean_strained_grouped_pos_cleangroup_vs_timed2['Group - Time Elapsed - Split'].apply(lambda x: x[1])
final_clean_strained_grouped_pos_cleangroup_vs_timed2['Group - Time Elapsed - Unit'] = final_clean_strained_grouped_pos_cleangroup_vs_timed2['Group - Time Elapsed - Unit'].apply(lambda x: "wks" if x == 'w' else x)
final_clean_strained_grouped_pos_cleangroup_vs_timed2['Group - Hours Elapsed (x7)'] = final_clean_strained_grouped_pos_cleangroup_vs_timed2.apply(lambda row: row['Group - Time Elapsed - Magn.']* hrs_week if row['Group - Time Elapsed - Unit'] == 'wks' else row['Group - Time Elapsed - Magn.'], axis = 1)
final_clean_strained_grouped_pos_cleangroup_vs_timed2

freqs = np.unique(final_clean_strained_grouped_pos_cleangroup_vs_timed2['Freq(kHz) (x1)'])
subs = np.unique(final_clean_strained_grouped_pos_cleangroup_vs_timed2['Subject'])
final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx = final_clean_strained_grouped_pos_cleangroup_vs_timed2.copy()
for freq in freqs:
    for sub in subs:
        mask = final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx[(final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx['Freq(kHz) (x1)'] == freq) & (final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx['Subject'] == sub)] # global for updates
        if len(mask) > 0:

            mask1 = final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx[(final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx['Freq(kHz) (x1)'] == freq) & (final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx['Subject'] == sub) & (final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx['vx (x4)'] == 'v1')]
            mask2 = final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx[(final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx['Freq(kHz) (x1)'] == freq) & (final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx['Subject'] == sub) & (final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx['vx (x4)'] == 'v2')]

            if not mask1.empty and not mask2.empty:
                mask1 = mask1.reset_index().iloc[0,:]
                mask2 = mask2.reset_index().iloc[0,:]

                total_syns = float(mask1['Synapses'] + mask2['Synapses'])
                total_ihcs = float(mask1['IHCs'] + mask2['IHCs'])

                if total_ihcs != 0:
                    ratio = total_syns / total_ihcs

                # if total_ihcs == 0 or total_syns == 0:
                #     print(sub, freq, total_syns, total_ihcs)
                # if total_syns == 0.0 or total_ihcs == 0.0:
                #     print(sub, freq)
                mask_index = mask.index
                final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx.loc[mask_index,'Synapse to IHC Ratio per Freq (y2)'] = ratio

## Waves from FLM

In [12]:
time_scale = 18
start_path = '/Users/leahashebir/Downloads/Manor_Practicum/liberman_data/abr_data/WPZ Electrophysiology'
subject_ABRs = {}

for subject in os.listdir(start_path):
    if subject in raw_synapse_counts['Subject'].values: # excluding subjects not in synapse count file
        for fq in os.listdir(os.path.join(start_path,subject)):
            if fq.startswith('ABR') and fq.endswith('.tsv'):
                match = re.search(r'-L-([\d.]+)\.tsv$', fq)
                if match:
                    freq = float(match.group(1))
                    if freq in raw_synapse_counts[raw_synapse_counts['Subject'] == subject]['Freq(kHz) (x1)'].values:
                        # if freq == 6.0 or freq == 7.0:
                        #     print(subject, freq)
                        # freqs.add(freq)
                        path = os.path.join(start_path,subject,fq)
                        data_df = read_custom_tsv(path)
                        if data_df['Freq(kHz)'].iloc[0] == freq:
                            subject_ABRs[(subject, freq)] = data_df
                        else:
                            print(f"Skipping subject {subject}, frequency {freq} due to mismatch.")

Skipping subject WPZ144, frequency 45.2 due to mismatch.
Skipping subject WPZ161, frequency 45.2 due to mismatch.
Skipping subject WPZ156, frequency 45.2 due to mismatch.
Skipping subject WPZ146, frequency 45.2 due to mismatch.
Skipping subject WPZ178, frequency 11.3 due to mismatch.
Skipping subject WPZ98, frequency 45.2 due to mismatch.
Skipping subject WPZ155, frequency 8.0 due to mismatch.


In [13]:
final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx.head()

,Subject,Latencies,Freq(kHz) (x1),Level(dB) (x2),Amplitude (x3),vx (x4),Strain (x5),Group - dB (x6),Group - Time Elapsed,Group,Synapses to IHC (y1),Synapses,IHCs,Group - Time Elapsed - Split,Group - Time Elapsed - Magn.,Group - Time Elapsed - Unit,Group - Hours Elapsed (x7),Synapse to IHC Ratio per Freq (y2)
0,WPZ145,"[337.5, 589.5, 724.5, 895.5, 1071.0]",45.2,70.0,0.033579,v1,C57B6,98,8wks,98dB 8wks post,8.750000,77.0,8.8,"[8, wks]",8,wks,1344,9.831461
0,WPZ145,"[337.5, 589.5, 724.5, 895.5, 1071.0]",45.2,70.0,0.033579,v2,C57B6,98,8wks,98dB 8wks post,10.888889,98.0,9,"[8, wks]",8,wks,1344,9.831461
1,WPZ145,"[266.40000000000003, 352.8, 525.6, 651.6, 824.4]",45.2,75.0,0.034262,v1,C57B6,98,8wks,98dB 8wks post,8.750000,77.0,8.8,"[8, wks]",8,wks,1344,9.831461
1,WPZ145,"[266.40000000000003, 352.8, 525.6, 651.6, 824.4]",45.2,75.0,0.034262,v2,C57B6,98,8wks,98dB 8wks post,10.888889,98.0,9,"[8, wks]",8,wks,1344,9.831461
2,WPZ145,"[252.0, 400.5, 715.5, 904.5, 1075.5]",45.2,80.0,0.154224,v1,C57B6,98,8wks,98dB 8wks post,8.750000,77.0,8.8,"[8, wks]",8,wks,1344,9.831461


In [14]:
penalty = L2Regularization(linear_operator=LinearDifferentialOperator(2))
shared_grid = np.linspace(0, time_scale, 244)
# basis = BSplineBasis(n_basis=7, domain_range=(0,18)) # old
# smoother = BasisSmoother(basis=basis, return_basis=True) # , regularization=penalty, smoothing_parameter=0.1, 
time_scale = 18
subjects = []
frequencies = []
levels = []
strains = []
amps = []
raw_waves = []
newbasis_waves = []
Xs = []
ys = []
fails = []
bases = []
all_latencies = []

for (sub, freq), df in subject_ABRs.items():
    for lvl in np.unique(df['Level(dB)']):
        latencies_series = final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx[(final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx['Subject'] == sub)\
            & (final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx['Freq(kHz) (x1)'] == freq)\
                & (final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx['Level(dB) (x2)'] == lvl)]\
            ['Latencies']
        
        if len(latencies_series) == 0:
            print(f'N/A latencies: ({sub}, {freq}, {lvl}) : {latencies_series}')
            continue

        latencies = latencies_series.values[0]
        latencies = [float(x) for x in latencies]
        all_latencies.append(latencies)
        # print(latencies)

    for lvl in np.unique(df['Level(dB)']):
        strain_series = final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx[(final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx['Subject'] == sub)\
            & (final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx['Freq(kHz) (x1)'] == freq)\
                & (final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx['Level(dB) (x2)'] == lvl)]\
            ['Strain (x5)']
        
        if len(strain_series) == 0:
            print(f'Strain error, none recorded: ({sub}, {freq}, {lvl}) : {strain_series}')
            continue

        amp_series = final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx[(final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx['Subject'] == sub)\
                & (final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx['Freq(kHz) (x1)'] == freq)]['Amplitude (x3)']
        if len(amp_series) == 0:
            print(f'Amplitude error, none recorded: ({sub}, {freq}, {lvl}) : {amp_series}')

        amp = float(amp_series.values[0])

        lvl = float(lvl)
        strain = strain_series.values[0]
        wave = full_interpolation(df, freq, lvl, time_scale)
        wave = np.asarray(wave, dtype=float)
        wave = wave.reshape(1, -1)
        
        grid = time_scale * np.arange(0, 244) / 244
        wave_input = skfda.FDataGrid(data_matrix=wave,grid_points=shared_grid)

        basis = BSplineBasis(domain_range=(latencies[0], latencies[-1]),knots = latencies, order = 3)
        bases.append(basis)
        smoother = BasisSmoother(basis=basis, return_basis=True, regularization=penalty, smoothing_parameter=1.0000e+02)

        wave_newbasis = smoother.fit_transform(wave_input)[0] # smoother will allow regularization for further tuning down the line...

        X = wave_newbasis.coefficients

        y_series = final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx[(final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx['Subject'] == sub)\
            & (final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx['Freq(kHz) (x1)'] == freq)\
            & (final_clean_strained_grouped_pos_cleangroup_vs_timed2_avgvx['Level(dB) (x2)'] == lvl)]\
            ['Synapse to IHC Ratio per Freq (y2)']
        
        # print(y_series)

        if len(y_series) == 0 or pd.isna(y_series.iloc[0]):
            print(f'N/A y: ({sub}, {freq}, {lvl})')
            continue

        y = float(y_series.iloc[0])

        # print((sub, freq, lvl, y))
        subjects.append(sub)
        frequencies.append(freq)
        levels.append(lvl)
        strains.append(strain)
        amps.append(amp)
        raw_waves.append(wave_input)
        newbasis_waves.append(wave_newbasis)
        Xs.append(X.flatten()) # used for model fitting, same as for OLS!
        ys.append(y)


final_waves_df_new_basis_best = pd.DataFrame(data = {'Subject' : subjects, 'Freq(kHz)' : frequencies, 'Level(dB)' : levels, 'Strain' : strains, 'Amplitude' : amps, 'Latencies' : all_latencies, 'Raw Waves' : raw_waves, 'Transformed Waves (X)' : Xs, 'Synapse to IHC Ratio per Freq (y2)' : ys})
final_waves_df_new_basis_best_clean = final_waves_df_new_basis_best.dropna().reset_index(drop=True)
final_waves_df_new_basis_best_clean.head()

,Subject,Freq(kHz),Level(dB),Strain,Amplitude,Latencies,Raw Waves,Transformed Waves (X),Synapse to IHC Ratio per Freq (y2)
0,WPZ145,45.2,70.0,C57B6,0.033579,"[337.5, 589.5, 724.5, 895.5, 1071.0]",[[[Data set: [[[0.08019387]\n [0.07692435]...,"[14.209802588706461, 22.79475268826122, 66.297...",9.831461
1,WPZ145,45.2,75.0,C57B6,0.033579,"[266.40000000000003, 352.8, 525.6, 651.6, 824.4]",[[[Data set: [[[0.48986208]\n [0.52359688]...,"[137.3503329972109, 221.51175318148637, 645.65...",9.831461
2,WPZ145,45.2,80.0,C57B6,0.033579,"[252.0, 400.5, 715.5, 904.5, 1075.5]",[[[Data set: [[[0.4259677 ]\n [0.39547801]...,"[-51.02658110433057, -82.50567816399604, -240....",9.831461
3,WPZ145,8.0,30.0,C57B6,0.154656,"[147.6, 298.8, 381.6, 583.2, 727.2]",[[[Data set: [[[0.76825543]\n [0.87247968]...,"[-17.086221830415422, -28.92681192644966, -68....",13.977273
4,WPZ145,8.0,35.0,C57B6,0.154656,"[198.0, 277.2, 417.6, 496.8, 594.0]",[[[Data set: [[[0. ]\n [0.09085721]...,"[-37.06828345795197, -62.302689010032395, -146...",13.977273


# 2. Image Embedding Pipeline

In [16]:
# Helper functions for embedding pipeline

def convert_max_proj_tensor(file_path, ch_order):
    start_t = time.time()
    last_t = start_t
    count = 0

    reader = aicsimageio.AICSImage(file_path)

    all_channels = []   

    for ch in range(reader.shape[1]):
        ch_name = ch_order[ch]
        
        channel_data = reader.get_image_data("CZYX", C=ch)
        channel_data = np.squeeze(channel_data, axis=0) 
        if ch_name == 'ctbp2':
            channel_data = normalize(channel_data.astype(np.uint16), maxval=(2**16-1)).astype(np.uint16)

        max_proj = np.max(channel_data, axis=0)
        all_channels.append(max_proj)

        # save_out(out, channel_data, ch_name, save2d=True)

        count += 1
        print("Image conversion time: "+str(time.time()-last_t)+" s")
        last_t = time.time()

    # Stack channels: list of (H, W) → (C, H, W)
    image_3d = np.stack(all_channels, axis=0)
    tensor = torch.from_numpy(image_3d).float()
                
    print("Total elapsed time: "+str(time.time()-start_t)+" s")
    return tensor


def normalize_per_channel(tensor):
    """
    Normalize each channel independently to [0, 1].
    
    Args:
        tensor: Shape [C, H, W]
    
    Returns:
        Normalized tensor of same shape
    """
    normalized = torch.zeros_like(tensor)
    
    for c in range(tensor.shape[0]):
        channel = tensor[c]
        normalized[c] = (channel - channel.min()) / (channel.max() - channel.min() + 1e-8)
    
    return normalized


def get_embedding_multi_channel(img_tensor, embedder, resize_to=(224, 224)):
    """
    Generate embedding from multi-channel image.
    
    Args:
        img_tensor: Tensor of shape [C, H, W]
        embedder: Multi-channel PyTorch model
        resize_to: Target size for model input
    
    Returns:
        Embedding vector of shape [embedding_dim]
    """
    transform = transforms.Compose([
        transforms.Resize(resize_to),
    ])
    
    # Add batch dimension: [C, H, W] → [1, C, H, W]
    img_batch = img_tensor.unsqueeze(0)
    img_resized = transform(img_batch)
    
    # Generate embedding
    with torch.no_grad():
        embedding = embedder(img_resized)
    
    # Remove batch dimension: [1, embedding_dim] → [embedding_dim]
    return embedding.squeeze(0)


def embedding_to_numpy(embedding_tensor):
    """
    Convert PyTorch tensor embedding to numpy array.
    
    Args:
        embedding_tensor: PyTorch tensor (can be on GPU or CPU)
    
    Returns:
        numpy array (always on CPU)
    """
    # Move to CPU if on GPU, then convert to numpy
    return embedding_tensor.cpu().detach().numpy()

In [17]:
def process_image_to_embedding(embedder, key, data_dir, ch_order, export_embeddings=False, convert_to_tensor=False):
    all_arrays = []
    for dir_name in os.listdir(data_dir):
            # check if dir_name is in key ID column
            if dir_name in key["ID"].values:
                group = key[key["ID"] == dir_name]["Group"].values[0]
            else:
                print(f"{dir_name} not found in key")
            
            for file_name in os.listdir(os.path.join(data_dir, dir_name)):
                if file_name.endswith(".lif"):
                    # freq = file_name.split(".")[6]+"."+file_name.split(".")[7]
                    # view = file_name.split(".")[8]

                    file_path = os.path.join(data_dir, dir_name, file_name)
                    tensor = convert_max_proj_tensor(file_path, ch_order)
                    normalized_tensor = normalize_per_channel(tensor)
                    embedding = get_embedding_multi_channel(normalized_tensor, embedder)
                    embedding_numpy = embedding_to_numpy(embedding)
                    all_arrays.append(embedding_numpy)

                    if export_embeddings:
                        np.save(f'embedding_v1.npy', embedding_numpy) #figure out naming convention
    
    all_embeddings = np.stack(all_arrays)
    
    if convert_to_tensor:
        torch_batch = torch.from_numpy(all_embeddings).float()
        print(f"Batch of embeddings{all_embeddings.shape} (batch_size=2, embedding_dim=512)\n[NOTE: Returned as torch tensor]")
        return torch_batch

    print(f"Batch of embeddings: {all_embeddings.shape} (batch_size=2, embedding_dim=512)")

    return all_embeddings
                # how to stack and export tensors effectively?

                # out_zarr = os.path.join("./zarrs", "liberman", group.replace(" ", "_"), f"{dir_name}_{freq}_{view}.zarr")
                # if not os.path.exists(os.path.join(out_zarr, 'ctbp2')):
                #     count += 1
                #     print(count, f"Saving {file_name} to {out_zarr}")
                #     out = zarr.open(out_zarr, 'a')
                
# np.load('.npy') to load later                    

## Complete Usage Example

Below is a complete workflow showing how to use all the components together.

In [19]:
def visualize_embeddings_2d(embeddings, labels=None, method='umap', title='Embedding Visualization',
                            colors=None, figsize=(10, 8), save_path=None):
    """
    Visualize high-dimensional embeddings in 2D.
    
    Args:
        embeddings: numpy array of shape [n_samples, embedding_dim]
        labels: Optional cluster labels or experimental groups
        method: 'umap' or 'tsne'
        title: Plot title
        colors: Optional custom colors for each point
        figsize: Figure size
        save_path: Optional path to save figure
    
    Returns:
        2D embedding coordinates
    """
    print(f"Reducing {embeddings.shape[1]}D embeddings to 2D using {method.upper()}...")
    
    if method == 'umap':
        reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
        embedding_2d = reducer.fit_transform(embeddings)
    elif method == 'tsne':
        reducer = TSNE(n_components=2, random_state=42, perplexity=30)
        embedding_2d = reducer.fit_transform(embeddings)
    else:
        raise ValueError(f"Unknown method: {method}")
    
    # Plot
    plt.figure(figsize=figsize)
    
    if labels is not None:
        unique_labels = np.unique(labels)
        cmap = plt.cm.get_cmap('tab10', len(unique_labels))
        
        for i, label in enumerate(unique_labels):
            mask = labels == label
            plt.scatter(embedding_2d[mask, 0], embedding_2d[mask, 1], 
                       c=[cmap(i)], label=f'Cluster {label}', 
                       alpha=0.7, s=50, edgecolors='black', linewidth=0.5)
        plt.legend()
    else:
        plt.scatter(embedding_2d[:, 0], embedding_2d[:, 1], 
                   c=colors, alpha=0.7, s=50, edgecolors='black', linewidth=0.5)
    
    plt.xlabel(f'{method.upper()} 1')
    plt.ylabel(f'{method.upper()} 2')
    plt.title(title)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved to {save_path}")
    
    plt.show()
    
    return embedding_2d


def visualize_reconstructions(model, dataloader, n_samples=5, model_type='vae', device='cpu'):
    """
    Visualize original vs reconstructed images.
    
    Args:
        model: Trained autoencoder or VAE
        dataloader: DataLoader with images
        n_samples: Number of samples to visualize
        model_type: 'autoencoder' or 'vae'
        device: 'cpu' or 'cuda'
    """
    model = model.to(device)
    model.eval()
    
    # Get batch of images
    for batch in dataloader:
        if isinstance(batch, (list, tuple)):
            images = batch[0]
        else:
            images = batch
        images = images.to(device)
        break
    
    # Reconstruct
    with torch.no_grad():
        if model_type == 'vae':
            reconstructions, mu, logvar = model(images)
        else:
            reconstructions, _ = model(images)
    
    # Move to CPU for visualization
    images = images.cpu().numpy()
    reconstructions = reconstructions.cpu().numpy()
    
    # Plot
    n_samples = min(n_samples, images.shape[0])
    fig, axes = plt.subplots(n_samples, 8, figsize=(20, 2.5*n_samples))
    
    channel_names = ['Myo7', 'GluR2', 'CtBP2', 'NF']
    
    for i in range(n_samples):
        for ch in range(4):
            # Original
            axes[i, ch].imshow(images[i, ch], cmap='gray')
            axes[i, ch].axis('off')
            if i == 0:
                axes[i, ch].set_title(f'Original\n{channel_names[ch]}')
            
            # Reconstructed
            axes[i, ch+4].imshow(reconstructions[i, ch], cmap='gray')
            axes[i, ch+4].axis('off')
            if i == 0:
                axes[i, ch+4].set_title(f'Reconstructed\n{channel_names[ch]}')
    
    plt.tight_layout()
    plt.suptitle('Original vs Reconstructed Images', y=1.01, fontsize=14)
    plt.show()


def plot_training_curves(losses, model_name='Model'):
    """
    Plot training loss curves.
    
    Args:
        losses: Loss dictionary or list from training
        model_name: Name for plot title
    """
    plt.figure(figsize=(10, 5))
    
    if isinstance(losses, dict):
        # VAE with multiple loss components
        plt.subplot(1, 2, 1)
        plt.plot(losses['total'], label='Total Loss')
        plt.plot(losses['recon'], label='Reconstruction')
        plt.plot(losses['kl'], label='KL Divergence')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title(f'{model_name} - Loss Components')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        plt.subplot(1, 2, 2)
        plt.plot(losses['total'])
        plt.xlabel('Epoch')
        plt.ylabel('Total Loss')
        plt.title(f'{model_name} - Total Loss')
        plt.grid(True, alpha=0.3)
    else:
        # Simple loss list
        plt.plot(losses)
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title(f'{model_name} - Training Loss')
        plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


def plot_cluster_quality(results):
    """
    Plot elbow curve and silhouette scores for cluster selection.
    
    Args:
        results: Dictionary from find_optimal_clusters()
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Elbow plot
    if len(results['inertias']) > 0:
        axes[0].plot(results['k_values'], results['inertias'], 'bo-')
        axes[0].set_xlabel('Number of Clusters (k)')
        axes[0].set_ylabel('Inertia')
        axes[0].set_title('Elbow Method')
        axes[0].grid(True, alpha=0.3)
    
    # Silhouette scores
    axes[1].plot(results['k_values'], results['silhouette_scores'], 'ro-')
    axes[1].set_xlabel('Number of Clusters (k)')
    axes[1].set_ylabel('Silhouette Score')
    axes[1].set_title('Silhouette Analysis')
    axes[1].axhline(y=0.5, color='g', linestyle='--', label='Good separation (>0.5)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Recommend best k
    best_k_idx = np.argmax(results['silhouette_scores'])
    best_k = results['k_values'][best_k_idx]
    best_score = results['silhouette_scores'][best_k_idx]
    
    print(f"\nRecommended k: {best_k} (silhouette score: {best_score:.3f})")


def visualize_latent_space_sampling(vae_model, n_samples=10, latent_dim=256, device='cpu'):
    """
    Sample random points from VAE latent space and decode.
    
    Useful for understanding what the VAE learned.
    """
    vae_model = vae_model.to(device)
    vae_model.eval()
    
    # Sample from standard normal
    z = torch.randn(n_samples, latent_dim).to(device)
    
    with torch.no_grad():
        generated = vae_model.decode(z)
    
    generated = generated.cpu().numpy()
    
    # Visualize
    fig, axes = plt.subplots(n_samples, 4, figsize=(12, 3*n_samples))
    channel_names = ['Myo7', 'GluR2', 'CtBP2', 'NF']
    
    for i in range(n_samples):
        for ch in range(4):
            axes[i, ch].imshow(generated[i, ch], cmap='gray')
            axes[i, ch].axis('off')
            if i == 0:
                axes[i, ch].set_title(f'{channel_names[ch]}')
    
    plt.tight_layout()
    plt.suptitle('Random Samples from VAE Latent Space', y=1.01, fontsize=14)
    plt.show()

## Visualization Tools

In [20]:
def cluster_embeddings(embeddings, method='kmeans', n_clusters=5, **kwargs):
    """
    Cluster embeddings using various algorithms.
    
    Args:
        embeddings: numpy array of shape [n_samples, embedding_dim]
        method: 'kmeans', 'dbscan', or 'hierarchical'
        n_clusters: Number of clusters (for kmeans/hierarchical)
        **kwargs: Additional arguments for clustering algorithm
    
    Returns:
        cluster_labels: Array of cluster assignments
        cluster_model: Fitted clustering model
    """
    if method == 'kmeans':
        from sklearn.cluster import KMeans
        model = KMeans(n_clusters=n_clusters, random_state=42, **kwargs)
        labels = model.fit_predict(embeddings)
        
    elif method == 'dbscan':
        from sklearn.cluster import DBSCAN
        eps = kwargs.get('eps', 0.5)
        min_samples = kwargs.get('min_samples', 5)
        model = DBSCAN(eps=eps, min_samples=min_samples)
        labels = model.fit_predict(embeddings)
        
    elif method == 'hierarchical':
        from sklearn.cluster import AgglomerativeClustering
        linkage = kwargs.get('linkage', 'ward')
        model = AgglomerativeClustering(n_clusters=n_clusters, linkage=linkage)
        labels = model.fit_predict(embeddings)
    
    else:
        raise ValueError(f"Unknown method: {method}")
    
    # Calculate silhouette score if we have multiple clusters
    n_unique = len(np.unique(labels))
    if n_unique > 1 and n_unique < len(embeddings):
        sil_score = silhouette_score(embeddings, labels)
        print(f"Silhouette Score: {sil_score:.3f}")
    else:
        print(f"Warning: Only {n_unique} unique cluster(s) found")
    
    return labels, model


def find_optimal_clusters(embeddings, max_k=10, method='kmeans'):
    """
    Find optimal number of clusters using elbow method and silhouette score.
    
    Args:
        embeddings: numpy array of embeddings
        max_k: Maximum number of clusters to try
        method: Clustering method
    
    Returns:
        Dictionary with scores for each k
    """
    results = {
        'k_values': [],
        'inertias': [],
        'silhouette_scores': []
    }
    
    for k in range(2, max_k + 1):
        labels, model = cluster_embeddings(embeddings, method=method, n_clusters=k)
        
        results['k_values'].append(k)
        
        if hasattr(model, 'inertia_'):
            results['inertias'].append(model.inertia_)
        
        if len(np.unique(labels)) > 1:
            sil = silhouette_score(embeddings, labels)
            results['silhouette_scores'].append(sil)
        else:
            results['silhouette_scores'].append(0)
        
        print(f"k={k}: Silhouette={results['silhouette_scores'][-1]:.3f}")
    
    return results


def analyze_clusters(embeddings, labels, metadata_df=None, metadata_cols=None):
    """
    Analyze cluster composition.
    
    Args:
        embeddings: numpy array of embeddings
        labels: Cluster labels
        metadata_df: DataFrame with metadata (subjects, freqs, groups, etc.)
        metadata_cols: List of column names to analyze
    
    Returns:
        Dictionary with cluster statistics
    """
    n_clusters = len(np.unique(labels))
    print(f"\n{'='*60}")
    print(f"CLUSTER ANALYSIS: {n_clusters} clusters found")
    print(f"{'='*60}\n")
    
    for cluster_id in np.unique(labels):
        mask = labels == cluster_id
        n_samples = np.sum(mask)
        
        print(f"Cluster {cluster_id}: {n_samples} samples ({100*n_samples/len(labels):.1f}%)")
        
        if metadata_df is not None and metadata_cols is not None:
            cluster_metadata = metadata_df[mask]
            
            for col in metadata_cols:
                if col in cluster_metadata.columns:
                    value_counts = cluster_metadata[col].value_counts()
                    print(f"  {col}: {dict(value_counts)}")
        
        print()
    
    return labels


def compare_clusters_to_experimental_groups(labels, group_labels, group_names=None):
    """
    Compare discovered clusters to known experimental groups.
    
    Args:
        labels: Cluster assignments
        group_labels: True experimental group labels
        group_names: Optional names for groups
    
    Returns:
        Confusion matrix and purity scores
    """
    from sklearn.metrics import confusion_matrix, adjusted_rand_score
    
    # Confusion matrix
    cm = confusion_matrix(group_labels, labels)
    
    # Adjusted Rand Index (similarity between clusterings)
    ari = adjusted_rand_score(group_labels, labels)
    
    print(f"Adjusted Rand Index: {ari:.3f}")
    print(f"(1.0 = perfect match, 0.0 = random, <0 = worse than random)")
    
    print(f"\nConfusion Matrix:")
    print(cm)
    
    return cm, ari

## Clustering & Analysis Utilities

In [21]:
class SynapseImageDataset(Dataset):
    """
    PyTorch Dataset for loading synapse .lif images.
    
    Integrates with existing convert_max_proj_tensor() and normalize_per_channel() functions.
    """
    def __init__(self, image_paths, ch_order=['myo7', 'glur2', 'ctbp2', 'nf'], 
                 augment=False, target_size=224):
        """
        Args:
            image_paths: List of paths to .lif files
            ch_order: Channel order for convert_max_proj_tensor
            augment: If True, apply random augmentations
            target_size: Resize images to this size
        """
        self.image_paths = image_paths
        self.ch_order = ch_order
        self.augment = augment
        self.target_size = target_size
        
        # Simple augmentations for contrastive learning
        if augment:
            self.aug_transform = transforms.Compose([
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomVerticalFlip(p=0.5),
                transforms.RandomRotation(degrees=15),
            ])
        else:
            self.aug_transform = None
        
        # Resize transform
        self.resize = transforms.Resize((target_size, target_size))
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        """
        Load and preprocess image.
        
        Returns:
            If augment=False: Single tensor [4, H, W]
            If augment=True: Tuple of (tensor1, tensor2) for contrastive learning
        """
        img_path = self.image_paths[idx]
        
        # Load image using existing function
        tensor = convert_max_proj_tensor(img_path, self.ch_order)
        
        # Normalize per channel
        tensor = normalize_per_channel(tensor)
        
        # Resize
        tensor = self.resize(tensor.unsqueeze(0)).squeeze(0)
        
        if self.augment:
            # Return two augmented views for contrastive learning
            view1 = self.aug_transform(tensor) if self.aug_transform else tensor
            view2 = self.aug_transform(tensor) if self.aug_transform else tensor
            return view1, view2
        else:
            return tensor


def create_dataloader(image_paths, batch_size=8, shuffle=True, augment=False, 
                      num_workers=0, target_size=224):
    """
    Create PyTorch DataLoader for synapse images.
    
    Args:
        image_paths: List of .lif file paths
        batch_size: Batch size for training
        shuffle: Shuffle data
        augment: Apply augmentations (for Shallow CNN training)
        num_workers: Number of parallel workers
        target_size: Image size
    
    Returns:
        DataLoader object
    """
    dataset = SynapseImageDataset(
        image_paths, 
        augment=augment,
        target_size=target_size
    )
    
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    return dataloader


def collect_all_lif_paths(data_dir, subjects=None):
    """
    Collect all .lif file paths from directory structure.
    
    Args:
        data_dir: Base directory (e.g., liberman_data/Confocal Data Charles Liberman/)
        subjects: Optional list of subject IDs to include (e.g., ['WPZ116', 'WPZ145'])
                  If None, include all subjects
    
    Returns:
        List of full paths to .lif files
    """
    lif_paths = []
    
    for subject_dir in os.listdir(data_dir):
        subject_path = os.path.join(data_dir, subject_dir)
        
        # Filter by subjects if specified
        if subjects is not None and subject_dir not in subjects:
            continue
        
        if os.path.isdir(subject_path):
            for file_name in os.listdir(subject_path):
                if file_name.endswith('.lif'):
                    full_path = os.path.join(subject_path, file_name)
                    lif_paths.append(full_path)
    
    print(f"Found {len(lif_paths)} .lif files")
    return lif_paths

## Data Loading & Preprocessing Utilities

In [22]:
class VAE(nn.Module):
    """
    Variational Autoencoder - BEST for unsupervised clustering.
    
    Why VAE is superior for this task:
    1. Structured latent space - regularization forces organization
    2. Natural clustering - similar images cluster together
    3. Probabilistic - captures uncertainty in embeddings
    4. Prevents overfitting - KL divergence acts as regularizer
    5. Interpretable - can sample and interpolate in latent space
    
    Architecture:
        Encoder: Conv layers → mean & log_variance
        Reparameterization: Sample from N(mean, variance)
        Decoder: Reconstruct from sampled latent vector
        Loss: Reconstruction + KL divergence
    """
    def __init__(self, input_channels=4, latent_dim=256):
        super(VAE, self).__init__()
        
        # ENCODER (same as Conv Autoencoder)
        self.enc1 = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )
        
        self.enc2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        
        self.enc3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        
        self.enc4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )
        
        self.flatten = nn.Flatten()
        
        # VAE-specific: Output mean and log_variance
        self.fc_mu = nn.Linear(256 * 14 * 14, latent_dim)
        self.fc_logvar = nn.Linear(256 * 14 * 14, latent_dim)
        
        # DECODER
        self.fc_dec = nn.Linear(latent_dim, 256 * 14 * 14)
        
        self.dec1 = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        
        self.dec2 = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        
        self.dec3 = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )
        
        self.dec4 = nn.Sequential(
            nn.ConvTranspose2d(32, input_channels, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )
        
        self.latent_dim = latent_dim
    
    def encode(self, x):
        """Encode to mean and log_variance"""
        x = self.enc1(x)
        x = self.enc2(x)
        x = self.enc3(x)
        x = self.enc4(x)
        x = self.flatten(x)
        
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        """
        Reparameterization trick: z = mu + epsilon * sigma
        where epsilon ~ N(0, 1)
        
        This allows backpropagation through sampling
        """
        std = torch.exp(0.5 * logvar)
        epsilon = torch.randn_like(std)
        z = mu + epsilon * std
        return z
    
    def decode(self, z):
        """Reconstruct image from latent vector"""
        x = self.fc_dec(z)
        x = x.view(-1, 256, 14, 14)
        x = self.dec1(x)
        x = self.dec2(x)
        x = self.dec3(x)
        x = self.dec4(x)
        return x
    
    def forward(self, x):
        """Full VAE forward pass"""
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z)
        return x_recon, mu, logvar


def vae_loss_function(recon_x, x, mu, logvar, beta=1.0):
    """
    VAE loss = Reconstruction loss + KL divergence
    
    Args:
        recon_x: Reconstructed images
        x: Original images
        mu: Mean of latent distribution
        logvar: Log variance of latent distribution
        beta: Weight for KL divergence (beta-VAE)
              - beta=1.0: Standard VAE
              - beta>1.0: Stronger regularization (more disentangled representations)
              - beta<1.0: Weaker regularization (better reconstructions)
    
    Returns:
        total_loss, recon_loss, kl_loss
    """
    # Reconstruction loss (MSE)
    recon_loss = F.mse_loss(recon_x, x, reduction='sum')
    
    # KL divergence: KL(N(mu, sigma) || N(0, 1))
    # = 0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    
    # Total loss
    total_loss = recon_loss + beta * kl_loss
    
    return total_loss, recon_loss, kl_loss


def train_vae(model, dataloader, epochs=30, lr=1e-3, beta=1.0, device='cpu'):
    """
    Train Variational Autoencoder.
    
    Args:
        beta: KL divergence weight
              - Start with beta=0.5 for easier training
              - Increase to beta=1.0-2.0 for better clustering
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    model.train()
    losses = {'total': [], 'recon': [], 'kl': []}
    
    for epoch in range(epochs):
        epoch_total = 0
        epoch_recon = 0
        epoch_kl = 0
        
        for batch_idx, batch in enumerate(dataloader):
            if isinstance(batch, (list, tuple)):
                images = batch[0]
            else:
                images = batch
            
            images = images.to(device)
            
            # Forward pass
            reconstructions, mu, logvar = model(images)
            
            # Compute loss
            total_loss, recon_loss, kl_loss = vae_loss_function(
                reconstructions, images, mu, logvar, beta=beta
            )
            
            # Backward pass
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()
            
            # Track losses
            batch_size = images.size(0)
            epoch_total += total_loss.item()
            epoch_recon += recon_loss.item()
            epoch_kl += kl_loss.item()
        
        # Average losses
        num_samples = len(dataloader.dataset)
        avg_total = epoch_total / num_samples
        avg_recon = epoch_recon / num_samples
        avg_kl = epoch_kl / num_samples
        
        losses['total'].append(avg_total)
        losses['recon'].append(avg_recon)
        losses['kl'].append(avg_kl)
        
        print(f"Epoch {epoch+1}/{epochs}, Total: {avg_total:.4f}, "
              f"Recon: {avg_recon:.4f}, KL: {avg_kl:.4f}")
    
    return losses


def extract_embeddings_vae(model, dataloader, use_mean=True, device='cpu'):
    """
    Extract embeddings from trained VAE.
    
    Args:
        use_mean: If True, use mu (deterministic)
                  If False, sample from distribution (stochastic)
    
    For clustering, use_mean=True is recommended for consistency.
    """
    model = model.to(device)
    model.eval()
    
    embeddings = []
    with torch.no_grad():
        for batch in dataloader:
            if isinstance(batch, (list, tuple)):
                batch = batch[0]
            batch = batch.to(device)
            
            mu, logvar = model.encode(batch)
            
            if use_mean:
                z = mu
            else:
                z = model.reparameterize(mu, logvar)
            
            embeddings.append(z.cpu().numpy())
    
    return np.vstack(embeddings)

## Phase 3: Variational Autoencoder (VAE) - RECOMMENDED

In [23]:
class ConvAutoencoder(nn.Module):
    """
    Symmetric convolutional autoencoder for unsupervised embedding learning.
    
    Better than Shallow CNN because:
    - Reconstruction objective forces meaningful representations
    - Deeper architecture learns hierarchical features
    - Bottleneck naturally creates compressed embeddings
    
    Architecture:
        Encoder: 4 conv blocks → 512-dim bottleneck
        Decoder: 4 transpose conv blocks → reconstruct 4 channels
        Loss: MSE reconstruction loss
    """
    def __init__(self, input_channels=4, latent_dim=512):
        super(ConvAutoencoder, self).__init__()
        
        # ENCODER
        # 224x224x4 → 112x112x32
        self.enc1 = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )
        
        # 112x112x32 → 56x56x64
        self.enc2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        
        # 56x56x64 → 28x28x128
        self.enc3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        
        # 28x28x128 → 14x14x256
        self.enc4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )
        
        # Flatten and bottleneck
        self.flatten = nn.Flatten()
        self.fc_enc = nn.Linear(256 * 14 * 14, latent_dim)
        
        # DECODER
        self.fc_dec = nn.Linear(latent_dim, 256 * 14 * 14)
        
        # 14x14x256 → 28x28x128
        self.dec1 = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        
        # 28x28x128 → 56x56x64
        self.dec2 = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        
        # 56x56x64 → 112x112x32
        self.dec3 = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )
        
        # 112x112x32 → 224x224x4
        self.dec4 = nn.Sequential(
            nn.ConvTranspose2d(32, input_channels, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()  # Output in [0,1] range
        )
        
        self.latent_dim = latent_dim
    
    def encode(self, x):
        """Extract embedding from image"""
        x = self.enc1(x)
        x = self.enc2(x)
        x = self.enc3(x)
        x = self.enc4(x)
        x = self.flatten(x)
        z = self.fc_enc(x)
        return z
    
    def decode(self, z):
        """Reconstruct image from embedding"""
        x = self.fc_dec(z)
        x = x.view(-1, 256, 14, 14)
        x = self.dec1(x)
        x = self.dec2(x)
        x = self.dec3(x)
        x = self.dec4(x)
        return x
    
    def forward(self, x):
        """Full forward pass: encode → decode"""
        z = self.encode(x)
        x_recon = self.decode(z)
        return x_recon, z


def train_conv_autoencoder(model, dataloader, epochs=20, lr=1e-3, device='cpu'):
    """
    Train Convolutional Autoencoder with MSE reconstruction loss.
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    model.train()
    losses = []
    
    for epoch in range(epochs):
        epoch_loss = 0
        for batch_idx, batch in enumerate(dataloader):
            # Handle paired data or single images
            if isinstance(batch, (list, tuple)):
                images = batch[0]
            else:
                images = batch
            
            images = images.to(device)
            
            # Forward pass
            reconstructions, embeddings = model(images)
            
            # Reconstruction loss
            loss = criterion(reconstructions, images)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(dataloader)
        losses.append(avg_loss)
        print(f"Epoch {epoch+1}/{epochs}, Reconstruction Loss: {avg_loss:.6f}")
    
    return losses


def extract_embeddings_autoencoder(model, dataloader, device='cpu'):
    """Extract embeddings from trained autoencoder"""
    model = model.to(device)
    model.eval()
    
    embeddings = []
    with torch.no_grad():
        for batch in dataloader:
            if isinstance(batch, (list, tuple)):
                batch = batch[0]
            batch = batch.to(device)
            z = model.encode(batch)
            embeddings.append(z.cpu().numpy())
    
    return np.vstack(embeddings)

## Phase 2: Convolutional Autoencoder

In [24]:
class ShallowCNN(nn.Module):
    """
    Lightweight 3-layer CNN for quick signal validation.
    
    Purpose: Fast check if images contain learnable morphological patterns
    Training time: ~5-10 minutes
    
    Architecture:
        Input: 4-channel 224x224 images
        Conv1: 4 → 32 channels (3x3)
        Conv2: 32 → 64 channels (3x3)
        Conv3: 64 → 128 channels (3x3)
        Global pooling → 256-dim embedding
    """
    def __init__(self, input_channels=4, embedding_dim=256):
        super(ShallowCNN, self).__init__()
        
        # Conv blocks
        self.conv1 = nn.Conv2d(input_channels, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)  # 224 → 112
        
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)  # 112 → 56
        
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2, 2)  # 56 → 28
        
        # Global average pooling + projection
        self.gap = nn.AdaptiveAvgPool2d(1)  # 28x28x128 → 1x1x128
        self.fc = nn.Linear(128, embedding_dim)
        
    def forward(self, x):
        # x: [batch, 4, 224, 224]
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool1(x)
        
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)
        
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool3(x)
        
        x = self.gap(x)  # [batch, 128, 1, 1]
        x = x.view(x.size(0), -1)  # [batch, 128]
        x = self.fc(x)  # [batch, embedding_dim]
        
        return x


def train_shallow_cnn_contrastive(model, dataloader, epochs=10, lr=1e-3, device='cpu'):
    """
    Train Shallow CNN with simple contrastive loss.
    
    Strategy: Augmented pairs of same image should have similar embeddings
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    model.train()
    losses = []
    
    for epoch in range(epochs):
        epoch_loss = 0
        for batch_idx, (img1, img2) in enumerate(dataloader):
            img1, img2 = img1.to(device), img2.to(device)
            
            # Get embeddings
            emb1 = model(img1)
            emb2 = model(img2)
            
            # Simple contrastive loss: minimize distance between augmented pairs
            loss = F.mse_loss(emb1, emb2)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(dataloader)
        losses.append(avg_loss)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")
    
    return losses


def extract_embeddings_shallow(model, dataloader, device='cpu'):
    """Extract embeddings for all images in dataloader"""
    model = model.to(device)
    model.eval()
    
    embeddings = []
    with torch.no_grad():
        for batch in dataloader:
            if isinstance(batch, (list, tuple)):
                batch = batch[0]  # Handle paired data
            batch = batch.to(device)
            emb = model(batch)
            embeddings.append(emb.cpu().numpy())
    
    return np.vstack(embeddings)

## Phase 1: Shallow CNN - Signal Validation

In [25]:
embedder =  # research and define this
key = pd.read_excel("/Users/leahashebir/Downloads/Manor_Practicum/liberman_data/WPZ Mouse groups.xlsx")
key['Group'] = key['Group'].str.strip()
key['Strain'] = key['Strain'].str.strip()
key.rename(columns={'ID#': 'ID'}, inplace=True)

data_dir = "/Users/leahashebir/Downloads/Manor_Practicum/liberman_data/Confocal Data Charles Liberman/"
ch_order = ['myo7', 'glur2', 'ctbp2', 'nf']

process_image_to_embedding(embedder, key, data_dir, ch_order, export_embeddings=True, convert_to_tensor=True)

SyntaxError: invalid syntax (824678630.py, line 1)

# Final Execution

In [26]:
# ============================================================================
# COMPLETE WORKFLOW EXAMPLE
# ============================================================================
# This cell demonstrates the full pipeline from data loading to clustering
# Uncomment and run sections as needed
# ============================================================================

"""
# STEP 1: Collect image paths
# ============================================================================
data_dir = "/Users/leahashebir/Downloads/Manor_Practicum/liberman_data/Confocal Data Charles Liberman/"

# Option A: Use all subjects
all_paths = collect_all_lif_paths(data_dir)

# Option B: Use specific subjects (recommended for testing)
test_subjects = ['WPZ116', 'WPZ145', 'WPZ174']
test_paths = collect_all_lif_paths(data_dir, subjects=test_subjects)

# Use a subset for quick testing
image_paths = test_paths[:20]  # First 20 images
print(f"Using {len(image_paths)} images for training")


# STEP 2: Create DataLoader
# ============================================================================
# Choose device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Create dataloader (no augmentation for autoencoder/VAE)
train_loader = create_dataloader(
    image_paths, 
    batch_size=4,  # Adjust based on your GPU memory
    shuffle=True,
    augment=False,  # Set True for Shallow CNN contrastive learning
    num_workers=0,  # Increase if using multiple CPU cores
    target_size=224
)


# STEP 3A: Train Shallow CNN (Quick Signal Check)
# ============================================================================
print("\\n" + "="*60)
print("PHASE 1: Shallow CNN Training")
print("="*60)

shallow_model = ShallowCNN(input_channels=4, embedding_dim=256)
print(f"Model parameters: {sum(p.numel() for p in shallow_model.parameters()):,}")

# If using contrastive learning, recreate dataloader with augmentation
# train_loader_aug = create_dataloader(image_paths, batch_size=4, augment=True)
# losses = train_shallow_cnn_contrastive(shallow_model, train_loader_aug, epochs=10, device=device)

# For now, skip Shallow CNN and go directly to VAE (more powerful)


# STEP 3B: Train Convolutional Autoencoder
# ============================================================================
print("\\n" + "="*60)
print("PHASE 2: Convolutional Autoencoder Training")
print("="*60)

autoencoder = ConvAutoencoder(input_channels=4, latent_dim=512)
print(f"Model parameters: {sum(p.numel() for p in autoencoder.parameters()):,}")

losses_ae = train_conv_autoencoder(
    autoencoder, 
    train_loader, 
    epochs=20,  # Increase for better results
    lr=1e-3,
    device=device
)

# Plot training curve
plot_training_curves(losses_ae, model_name='Convolutional Autoencoder')

# Visualize reconstructions
visualize_reconstructions(autoencoder, train_loader, n_samples=3, 
                         model_type='autoencoder', device=device)


# STEP 3C: Train VAE (RECOMMENDED)
# ============================================================================
print("\\n" + "="*60)
print("PHASE 3: Variational Autoencoder Training")
print("="*60)

vae = VAE(input_channels=4, latent_dim=256)
print(f"Model parameters: {sum(p.numel() for p in vae.parameters()):,}")

losses_vae = train_vae(
    vae, 
    train_loader, 
    epochs=30,  # Increase to 50-100 for best results
    lr=1e-3,
    beta=0.5,  # Start with lower beta, increase gradually
    device=device
)

# Plot training curves
plot_training_curves(losses_vae, model_name='VAE')

# Visualize reconstructions
visualize_reconstructions(vae, train_loader, n_samples=3, 
                         model_type='vae', device=device)

# Visualize random samples from latent space
visualize_latent_space_sampling(vae, n_samples=5, latent_dim=256, device=device)


# STEP 4: Extract Embeddings
# ============================================================================
print("\\n" + "="*60)
print("Extracting Embeddings")
print("="*60)

# Choose your best model (VAE recommended)
best_model = vae

# Extract embeddings
embeddings = extract_embeddings_vae(best_model, train_loader, use_mean=True, device=device)
print(f"Extracted embeddings shape: {embeddings.shape}")


# STEP 5: Clustering
# ============================================================================
print("\\n" + "="*60)
print("Clustering Analysis")
print("="*60)

# Find optimal number of clusters
results = find_optimal_clusters(embeddings, max_k=10, method='kmeans')
plot_cluster_quality(results)

# Use recommended k or choose your own
k_optimal = 5  # Adjust based on plot

# Perform clustering
cluster_labels, cluster_model = cluster_embeddings(
    embeddings, 
    method='kmeans',  # or 'dbscan', 'hierarchical'
    n_clusters=k_optimal
)

# Analyze clusters (if you have metadata)
# metadata_df = pd.DataFrame({'Subject': [...], 'Freq': [...], 'Group': [...]})
# analyze_clusters(embeddings, cluster_labels, metadata_df, 
#                 metadata_cols=['Subject', 'Group'])


# STEP 6: Visualization
# ============================================================================
print("\\n" + "="*60)
print("Visualizing Embeddings")
print("="*60)

# UMAP visualization
embedding_2d_umap = visualize_embeddings_2d(
    embeddings, 
    labels=cluster_labels,
    method='umap',
    title='VAE Embeddings - UMAP Projection',
    save_path='embeddings_umap.png'
)

# t-SNE visualization
embedding_2d_tsne = visualize_embeddings_2d(
    embeddings, 
    labels=cluster_labels,
    method='tsne',
    title='VAE Embeddings - t-SNE Projection',
    save_path='embeddings_tsne.png'
)


# STEP 7: Compare to Experimental Groups (Optional)
# ============================================================================
# If you have ground truth experimental groups:
# true_groups = [...]  # Array of true group labels
# cm, ari = compare_clusters_to_experimental_groups(cluster_labels, true_groups)


# STEP 8: Save Model and Embeddings
# ============================================================================
print("\\n" + "="*60)
print("Saving Results")
print("="*60)

# Save trained model
torch.save(vae.state_dict(), 'vae_synapse_model.pth')
print("Model saved to vae_synapse_model.pth")

# Save embeddings
np.save('synapse_embeddings.npy', embeddings)
np.save('cluster_labels.npy', cluster_labels)
print("Embeddings and labels saved")

print("\\n" + "="*60)
print("PIPELINE COMPLETE!")
print("="*60)
"""

# Remove triple quotes above to run the complete workflow

'\n# STEP 1: Collect image paths\n# ============================================================================\ndata_dir = "/Users/leahashebir/Downloads/Manor_Practicum/liberman_data/Confocal Data Charles Liberman/"\n\n# Option A: Use all subjects\nall_paths = collect_all_lif_paths(data_dir)\n\n# Option B: Use specific subjects (recommended for testing)\ntest_subjects = [\'WPZ116\', \'WPZ145\', \'WPZ174\']\ntest_paths = collect_all_lif_paths(data_dir, subjects=test_subjects)\n\n# Use a subset for quick testing\nimage_paths = test_paths[:20]  # First 20 images\nprint(f"Using {len(image_paths)} images for training")\n\n\n# STEP 2: Create DataLoader\n# ============================================================================\n# Choose device\ndevice = \'cuda\' if torch.cuda.is_available() else \'cpu\'\nprint(f"Using device: {device}")\n\n# Create dataloader (no augmentation for autoencoder/VAE)\ntrain_loader = create_dataloader(\n    image_paths, \n    batch_size=4,  # Adjust based o

The history saving thread hit an unexpected error (OperationalError('unable to open database file')).History will not be written to the database.
